In [61]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler ,LabelEncoder,OneHotEncoder
import pickle

In [62]:
df = pd.read_csv('Churn_Modelling.csv')

In [63]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [64]:
df.isnull().sum()

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [65]:
df.drop(['RowNumber','CustomerId','Surname'],inplace=True,axis=1)

In [66]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [67]:
lable_encoder_gender = LabelEncoder()
df['Gender']=lable_encoder_gender.fit_transform(df['Gender'])
ohe = OneHotEncoder()

In [68]:
geo_encoded=ohe.fit_transform(df[['Geography']]).toarray()

In [69]:
ohe.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [70]:
df_2 = pd.DataFrame(geo_encoded,columns=ohe.get_feature_names_out(['Geography']))

In [71]:
df_2

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [72]:
df = pd.concat([df.drop('Geography',axis=1),df_2],axis=1)

In [73]:
df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [142]:
## save The encoder 
with open('lable_encoder_gender.pkl','wb') as file:
    pickle.dump(lable_encoder_gender,file)

with open('OneHotEncoder_geo.pkl','wb') as file2:
    pickle.dump(ohe,file2)

In [75]:
## Dived dataset into dependent and independent feature
X= df.drop('Exited',axis=1)
y = df['Exited']

In [76]:
X.shape

(10000, 12)

In [77]:
## Train test split
x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=42)

In [78]:
## scalling
ststd_sclr = StandardScaler()

In [79]:

x_train_scaled = ststd_sclr.fit_transform(x_train)

x_test_sclaed = ststd_sclr.transform(x_test)

In [80]:
with open('scaler.pkl','wb')as file3:
    pickle.dump(ststd_sclr,file3)

## ___***ANN Implementation***___

In [81]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [82]:
x_train.shape[1],

(12,)

In [83]:
## Build ANN Model
model = Sequential([
    Dense(64,activation='relu',input_shape=(x_train.shape[1],)), ## first hidden layer
    Dense(32,activation='relu'),#hl2
    Dense(1,activation='sigmoid')#output layer

]
)

c:\Users\maith\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [84]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [85]:
import tensorflow
opt = tensorflow.keras.optimizers.Adam(learning_rate=0.01)
los =tensorflow.keras.losses.BinaryCrossentropy()


In [86]:
## compile the complile
model.compile(optimizer=opt,loss=los,metrics=['accuracy'])

In [131]:
## setup tensorbord
logdir = 'log/fit/'+ datetime.datetime.now().strftime('%Y%m%d-%H%M%S')

In [132]:
tensorflow_callback = TensorBoard(log_dir=logdir,histogram_freq=1)

In [133]:
## Setup Early Stoping
Early_stopping_claa = EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [135]:
## Train model
history = model.fit(
    x_train_scaled,y_train,validation_data = (x_test_sclaed,y_test),epochs=100,
    callbacks = [tensorflow_callback, Early_stopping_claa]
    )

Epoch 1/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8656 - loss: 0.3233 - val_accuracy: 0.8548 - val_loss: 0.3510
Epoch 2/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8660 - loss: 0.3227 - val_accuracy: 0.8596 - val_loss: 0.3516
Epoch 3/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8680 - loss: 0.3211 - val_accuracy: 0.8552 - val_loss: 0.3482
Epoch 4/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8673 - loss: 0.3187 - val_accuracy: 0.8588 - val_loss: 0.3523
Epoch 5/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8664 - loss: 0.3165 - val_accuracy: 0.8552 - val_loss: 0.3488
Epoch 6/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8701 - loss: 0.3134 - val_accuracy: 0.8568 - val_loss: 0.3545
Epoch 7/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8695 - loss: 0.3106 - val_accuracy: 0.8572 - val_loss: 0.3597
Epoch 8/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8693 - loss: 0.3094 - val_accu

In [136]:
model.save('model.h5')

In [138]:
## load TenserBoard extenson
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [141]:
%tensorboard --logdir log/fit

Reusing TensorBoard on port 6010 (pid 20184), started 0:00:02 ago. (Use '!kill 20184' to kill it.)